In [0]:
CATALOG   = "hindsight_dev"
LANDING   = "https://www.sec.gov/data-research/sec-markets-data/financial-statement-data-sets"
VOLUME    = f"/Volumes/{CATALOG}/bronze/raw"
CONTROL   = f"{CATALOG}.ops.ingest_control"
HEADERS = {"User-Agent": "Utkarsh Saraogi utkarshsaraogi2000@gmail.com"}

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTROL} (
    quarter          STRING  COMMENT 'e.g. 2024q1',
    zip_url          STRING,
    status           STRING  COMMENT 'PENDING | RUNNING | DONE | FAILED',
    bytes_downloaded BIGINT,
    files_extracted  INT,
    row_counts       MAP<STRING, BIGINT>,
    attempts         INT,
    last_error       STRING,
    discovered_at    TIMESTAMP,
    completed_at     TIMESTAMP
)
USING DELTA
COMMENT 'One row per SEC quarterly drop. Drives idempotent backfill.'
""")
print("control table ready")

In [0]:
import requests, re
from datetime import datetime
from pyspark.sql import functions as F, types as T

r = requests.get(LANDING, headers=HEADERS, timeout=60)
r.raise_for_status()

hrefs = re.findall(r'href="([^"]+\.zip)"', r.text)
urls  = sorted({h if h.startswith("http") else "https://www.sec.gov" + h for h in hrefs})

rows = []
for u in urls:
    m = re.search(r'(\d{4}q[1-4])\.zip$', u)
    if m:
        rows.append((m.group(1), u))

if not rows:
    raise RuntimeError("No quarters discovered -- SEC page structure may have changed.")

schema = T.StructType([
    T.StructField("quarter", T.StringType()),
    T.StructField("zip_url", T.StringType()),
])
df = (spark.createDataFrame(rows, schema)
        .withColumn("status", F.lit("PENDING"))
        .withColumn("attempts", F.lit(0))
        .withColumn("discovered_at", F.current_timestamp()))

df.createOrReplaceTempView("discovered")

# MERGE so re-running never resets completed quarters
spark.sql(f"""
MERGE INTO {CONTROL} t
USING discovered s
ON t.quarter = s.quarter
WHEN NOT MATCHED THEN INSERT (quarter, zip_url, status, attempts, discovered_at)
VALUES (s.quarter, s.zip_url, s.status, s.attempts, s.discovered_at)
""")

print(f"discovered {len(rows)} quarters")
display(spark.sql(f"SELECT status, count(*) n FROM {CONTROL} GROUP BY status"))

In [0]:
pending = [r.quarter for r in spark.sql(
    f"SELECT quarter FROM {CONTROL} WHERE status <> 'DONE' ORDER BY quarter"
).collect()]

print(f"{len(pending)} pending")
dbutils.jobs.taskValues.set(key="quarters", value=pending)